In [ ]:
!nvidia-smi

In [ ]:
!pip install -q vllm lm-format-enforcer pandas datasets scikit-learn

In [ ]:
!pip install "protobuf==5.29.5"

In [ ]:
!pip install -q peft trl bitsandbytes accelerate

In [ ]:
!pip install -q langdetect nltk immutabledict spacy emoji syllapy unicodedata2 absl-py
!python -m spacy download en_core_web_sm

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
import os
import gc
import ast
import json
import time
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from collections import Counter, defaultdict

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports OK")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Llama-3-8B LoRA Fine-Tuning via Qwen3.5-9B Distillation on IF Training Data

**Goal:** Fine-tune `Meta-Llama-3-8B-Instruct` using QLoRA on instruction-following data distilled from local `Qwen3.5-9B` inference.

**Pipeline:**
1. Load `allenai/IF_multi_constraints_upto5` training data (strictly excluding IFBench test)
2. Generate teacher responses with local Qwen3.5-9B via vLLM
3. Validate teacher responses against constraint verification functions
4. Create filtered SFT dataset from validated (prompt, response) pairs
5. QLoRA fine-tune Llama-3-8B-Instruct on the distilled dataset
6. Evaluate on IFBench test set using the official evaluation harness

**Hardware:** NVIDIA L4 GPU (24 GB VRAM) — Google Colab

**References:**
- [IFBench paper](https://arxiv.org/html/2507.02833v1) (Pyatkin et al., NeurIPS 2025 D&B)
- [IFBench repo](https://github.com/allenai/IFBench)
- [IF_multi_constraints_upto5](https://huggingface.co/datasets/allenai/IF_multi_constraints_upto5)

In [ ]:
# ===================== Configuration =====================

# Teacher model (local inference via vLLM)
TEACHER_MODEL = "Qwen/Qwen3.5-9B"
TEACHER_MAX_TOKENS = 2048
TEACHER_TEMPERATURE = 0.0
TEACHER_MAX_MODEL_LEN = 8192

# Student model
STUDENT_MODEL = "meta-llama/Meta-Llama-3-8B-Instruct"

# Dataset
TRAIN_SUBSET_SIZE = 1000
TRAIN_SEED = 42
MIN_RESPONSE_WORDS = 20

# QLoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]
USE_4BIT = True

# SFT Training
NUM_TRAIN_EPOCHS = 3
PER_DEVICE_TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 2048
WARMUP_RATIO = 0.03
LR_SCHEDULER = "cosine"

# Evaluation
EVAL_MAX_TOKENS = 2048

print("=== Configuration ===")
for _name, _val in list(locals().items()):
    if _name.isupper() and not _name.startswith("_"):
        print(f"  {_name}: {_val}")

## 2. Dataset Loading & Subset Sampling

Load the AllenAI IF-RLVR training data (`allenai/IF_multi_constraints_upto5`, ~95k examples with up to 5 IFEval/IFBench-Train constraints each) and sample a clean subset with **no overlap** with IFBench test prompts.

In [ ]:
print("Loading allenai/IF_multi_constraints_upto5...")
full_ds = load_dataset("allenai/IF_multi_constraints_upto5", split="train")

print(f"Total examples: {len(full_ds):,}")
print(f"Columns: {full_ds.column_names}")

print(f"\nSource dataset distribution:")
for k, v in sorted(Counter(full_ds["dataset"]).items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:,}")

print(f"\nConstraint type distribution:")
for k, v in sorted(Counter(full_ds["constraint_type"]).items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:,}")

print(f"\n--- Sample example ---")
ex = full_ds[0]
for k in ex:
    print(f"  {k}: {str(ex[k])[:200]}...")

In [ ]:
def extract_user_prompt(example):
    """Extract the user message content from the messages field."""
    msgs = example["messages"]
    if isinstance(msgs, str):
        msgs = json.loads(msgs)
    for msg in msgs:
        if isinstance(msg, dict) and msg.get("role") == "user":
            return msg["content"]
    return None


def parse_ground_truth(gt_str):
    """Parse ground_truth string -> (instruction_id_list, kwargs_list)."""
    try:
        gt = ast.literal_eval(gt_str)
        if isinstance(gt, list) and len(gt) > 0:
            entry = gt[0]
            return entry.get("instruction_id", []), entry.get("kwargs", [])
    except Exception:
        pass
    return [], []


# Load IFBench test set to prevent any train/test contamination
print("Loading IFBench test set for contamination exclusion...")
ifbench_test = load_dataset("allenai/IFBench_test", split="train")
test_prompt_prefixes = {ex["prompt"].strip().lower()[:300] for ex in ifbench_test}
print(f"IFBench test prompts loaded: {len(ifbench_test)}")

# Filter: keep only examples with no test overlap and a valid extractable prompt
clean_indices = []
for i in range(len(full_ds)):
    prompt = extract_user_prompt(full_ds[i])
    if prompt is None:
        continue
    if prompt.strip().lower()[:300] in test_prompt_prefixes:
        continue
    clean_indices.append(i)

print(f"Clean training candidates: {len(clean_indices):,} / {len(full_ds):,}")

# Random sample
rng = random.Random(TRAIN_SEED)
sample_indices = sorted(rng.sample(clean_indices, min(TRAIN_SUBSET_SIZE, len(clean_indices))))
train_subset = full_ds.select(sample_indices)

print(f"\nSampled {len(train_subset):,} training examples")
print("Source distribution in sample:")
for k, v in sorted(Counter(train_subset["dataset"]).items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

In [ ]:
# --- Output directory (Google Drive for persistence) ---
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/Colab_Data/LoRA/IFBench/llama3_8b_lora_qwen35_distill"
else:
    SAVE_DIR = os.path.abspath("./llama3_8b_lora_qwen35_distill")

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

## 3. Qwen3.5-9B Teacher Loading & Inference

Load the teacher model locally using vLLM on the L4 GPU.  
Generations are cached to Google Drive so the notebook can be restarted without re-running inference.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
if __name__ == '__main__':
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    print(f"Loading tokenizer for {TEACHER_MODEL}...")
    teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL, use_fast=True)

    print(f"Loading {TEACHER_MODEL} with vLLM...")
    print("  PagedAttention: enabled (automatic)")
    print("  Cont. batching: enabled (automatic)")

    teacher_llm = LLM(
        model=TEACHER_MODEL,
        dtype="float16",
        gpu_memory_utilization=0.95,
        max_model_len=TEACHER_MAX_MODEL_LEN,
        enforce_eager=False,
    )

    teacher_sampling = SamplingParams(
        temperature=TEACHER_TEMPERATURE,
        top_k=20,
        max_tokens=TEACHER_MAX_TOKENS,
    )

    print("Teacher model loaded!")

In [ ]:
TEACHER_CACHE = os.path.join(SAVE_DIR, "teacher_generations.json")
TEACHER_CKPT  = os.path.join(SAVE_DIR, "teacher_checkpoint.jsonl")

teacher_prompts = []
teacher_meta = []

for i, example in enumerate(train_subset):
    user_prompt = extract_user_prompt(example)
    if user_prompt is None:
        continue

    instruction_ids, kwargs_list = parse_ground_truth(example["ground_truth"])

    formatted = teacher_tokenizer.apply_chat_template(
        [{"role": "user", "content": user_prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )

    prompt_tok_len = len(teacher_tokenizer.encode(formatted))
    max_input_len = TEACHER_MAX_MODEL_LEN - TEACHER_MAX_TOKENS
    if prompt_tok_len > max_input_len:
        continue

    teacher_prompts.append(formatted)
    teacher_meta.append({
        "idx": i,
        "key": example["key"],
        "user_prompt": user_prompt,
        "instruction_ids": instruction_ids,
        "kwargs_list": kwargs_list,
        "dataset_source": example["dataset"],
        "constraint": example["constraint"],
    })

skipped = len(train_subset) - len(teacher_prompts)
print(f"Prepared {len(teacher_prompts)} prompts for teacher inference")
print(f"  Skipped {skipped} prompts exceeding context limit "
      f"({TEACHER_MAX_MODEL_LEN} - {TEACHER_MAX_TOKENS} = {TEACHER_MAX_MODEL_LEN - TEACHER_MAX_TOKENS} max input tokens)")
if teacher_prompts:
    print(f"\nExample prompt (first 400 chars):\n{teacher_prompts[0][:400]}")

In [ ]:
if os.path.exists(TEACHER_CACHE):
    print(f"Loading cached teacher generations from:\n  {TEACHER_CACHE}")
    with open(TEACHER_CACHE) as f:
        teacher_results = json.load(f)
    print(f"Loaded {len(teacher_results)} cached results")
else:
    # Resume from partial checkpoint if available
    done_keys = set()
    partial_results = []
    if os.path.exists(TEACHER_CKPT):
        with open(TEACHER_CKPT) as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    partial_results.append(rec)
                    done_keys.add(rec["key"])
                except json.JSONDecodeError:
                    continue
        print(f"Resuming from checkpoint: {len(partial_results)} already done")

    remaining_indices = [i for i, m in enumerate(teacher_meta) if m["key"] not in done_keys]
    remaining_prompts = [teacher_prompts[i] for i in remaining_indices]

    if remaining_prompts:
        print(f"Generating teacher responses for {len(remaining_prompts)} prompts...")
        gen_start = time.time()
        outputs = teacher_llm.generate(remaining_prompts, teacher_sampling)
        gen_time = time.time() - gen_start

        total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
        throughput = total_tokens / gen_time if gen_time > 0 else 0

        print(f"\nGeneration complete:")
        print(f"  Time:       {gen_time / 60:.1f} min")
        print(f"  Tokens:     {total_tokens:,}")
        print(f"  Throughput: {throughput:.1f} tok/s")

        with open(TEACHER_CKPT, "a") as f:
            for j, o in enumerate(outputs):
                orig_idx = remaining_indices[j]
                meta = teacher_meta[orig_idx]
                rec = {
                    **meta,
                    "response": o.outputs[0].text.strip(),
                    "n_tokens": len(o.outputs[0].token_ids),
                }
                f.write(json.dumps(rec) + "\n")
                partial_results.append(rec)
    else:
        print("All prompts already generated (from checkpoint)")

    teacher_results = partial_results

    with open(TEACHER_CACHE, "w") as f:
        json.dump(teacher_results, f)
    print(f"All {len(teacher_results)} generations saved to:\n  {TEACHER_CACHE}")

# Stats
lengths = [len(r["response"].split()) for r in teacher_results]
print(f"\nTeacher response stats ({len(teacher_results)} total):")
print(f"  Mean words:   {np.mean(lengths):.0f}")
print(f"  Median words: {int(np.median(lengths))}")
print(f"  Range:        [{min(lengths)}, {max(lengths)}]")

In [ ]:
# Free teacher model GPU memory before validation & student training
for var_name in ["teacher_llm", "teacher_tokenizer", "outputs"]:
    if var_name in dir():
        exec(f"del {var_name}")
gc.collect()
torch.cuda.empty_cache()

print("GPU memory after teacher cleanup:")
print(f"  Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"  Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

## 4. Response Validation & Filtering

Use the official IFBench constraint verification functions to validate teacher outputs.  
Only responses where **all verifiable constraints pass** are kept for training.  
Constraints not found in the registry (e.g., IFBench-Train types not in the evaluation repo) are marked as unverifiable and are not penalized.

In [ ]:
# ── Repo setup ───────────────────────────────────────────────────────────────
# Two separate repos are needed:
#
#   allenai/IFBench         → 58 IFBench TEST constraint verifiers + evaluation_lib
#   allenai/open-instruct   → IFEvalG: 25 IFEval + 29 IFTrain TRAINING constraint verifiers
#
# The training data (IF_multi_constraints_upto5) uses IFEval + IFTrain types.
# The IFBench repo only has test types — hence 100% unverified without IFEvalG.

import sys
import importlib
import importlib.util

# Clone IFBench (test constraint verifiers + evaluation harness)
!rm -rf /content/IFBench
!git clone --depth 1 https://github.com/allenai/IFBench.git /content/IFBench

# Sparse-clone open-instruct — only fetch the IFEvalG subdirectory (~few MB)
!rm -rf /content/open-instruct
!git clone --depth 1 --filter=blob:none --sparse \
    https://github.com/allenai/open-instruct.git /content/open-instruct
!cd /content/open-instruct && \
    git sparse-checkout init --cone && \
    git sparse-checkout set open_instruct/IFEvalG


# ── Step 1: Scrub sys.path and sys.modules of any stale state ────────────────
# If this cell ran before, IFEvalG paths may still be in sys.path and Python
# will resolve the WRONG instructions_registry.py when we try to load IFBench.
_stale_paths = [p for p in sys.path
                if "open-instruct" in p or "IFEvalG" in p]
for p in _stale_paths:
    sys.path.remove(p)

for mod_name in list(sys.modules):
    if any(tag in mod_name for tag in
           ["instructions_registry", "instructions", "evaluation_lib",
            "open_instruct", "IFEvalG", "_instructions_registry"]):
        del sys.modules[mod_name]


# ── Step 2: Load IFBench test registry + evaluation_lib ──────────────────────
# IFBench MUST be loaded first, before any IFEvalG paths are on sys.path,
# so `import instructions_registry` resolves to the IFBench version.
if "/content/IFBench" not in sys.path:
    sys.path.insert(0, "/content/IFBench")

import instructions_registry as _ifbench_reg
import evaluation_lib

IFBENCH_REGISTRY = dict(_ifbench_reg.INSTRUCTION_DICT)
print(f"IFBench test registry: {len(IFBENCH_REGISTRY)} types")


# ── Step 3: Set up open_instruct package so IFEvalG can import itself ────────
IFEVALG_DIR = "/content/open-instruct/open_instruct/IFEvalG"
_oi_root = "/content/open-instruct"
_oi_pkg  = os.path.join(_oi_root, "open_instruct")

os.makedirs(_oi_pkg, exist_ok=True)
for _init in [os.path.join(_oi_pkg, "__init__.py"),
              os.path.join(IFEVALG_DIR, "__init__.py")]:
    if not os.path.exists(_init):
        open(_init, "w").close()

if _oi_root not in sys.path:
    sys.path.insert(0, _oi_root)


# ── Step 4: Load IFEvalG training registry via importlib ─────────────────────
# Uses spec_from_file_location with a unique module name so it never collides
# with the IFBench instructions_registry already in sys.modules.
def _load_registry(repo_dir, label):
    """Load INSTRUCTION_DICT from an instructions_registry.py in repo_dir."""
    reg_file = os.path.join(repo_dir, "instructions_registry.py")
    if not os.path.isfile(reg_file):
        print(f"  [{label}] instructions_registry.py not found in {repo_dir}")
        return {}
    unique_name = f"_instructions_registry_{label}"
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
    spec = importlib.util.spec_from_file_location(unique_name, reg_file)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[unique_name] = mod
    try:
        spec.loader.exec_module(mod)
    except Exception as e:
        print(f"  [{label}] Failed to exec module: {e}")
        return {}
    registry = dict(getattr(mod, "INSTRUCTION_DICT", {}))
    print(f"  [{label}] Loaded {len(registry)} constraint types")
    return registry

IFEVALG_REGISTRY = _load_registry(IFEVALG_DIR, "ifevalg")


# ── Step 5: Merge registries ─────────────────────────────────────────────────
# IFBench entries take precedence if IDs overlap.
CONSTRAINT_REGISTRY = {**IFEVALG_REGISTRY, **IFBENCH_REGISTRY}

n_overlap = len(set(IFEVALG_REGISTRY) & set(IFBENCH_REGISTRY))
print(f"\nCombined constraint registry: {len(CONSTRAINT_REGISTRY)} types")
print(f"  IFEvalG (IFEval+IFTrain): {len(IFEVALG_REGISTRY)}")
print(f"  IFBench (test only):      {len(IFBENCH_REGISTRY)}")
print(f"  Overlap:                  {n_overlap}")
print(f"\nSample IDs: {list(CONSTRAINT_REGISTRY.keys())[:10]}")

In [ ]:
VALIDATION_CACHE = os.path.join(SAVE_DIR, "validation_results.json")


def validate_response(result):
    """Validate a teacher response against its constraints.

    Returns a dict with:
      - status: 'passed' | 'failed' | 'unverified' | 'empty'
      - n_total, n_verified, n_passed
      - details: per-constraint verification

    Status logic:
      'passed'     — all verifiable constraints satisfied
      'failed'     — at least one verifiable constraint NOT satisfied
      'unverified' — no constraint in the registry could be checked (neutral)
      'empty'      — response is blank

    Exceptions during check_following are treated as 'unverified' (not 'failed')
    so mismatched kwargs from IFTrain variants don't unfairly penalize examples.
    """
    response = result["response"]
    instruction_ids = result["instruction_ids"]
    kwargs_list = result["kwargs_list"]
    user_prompt = result["user_prompt"]

    if not response or not response.strip():
        return {"status": "empty", "n_total": len(instruction_ids),
                "n_verified": 0, "n_passed": 0, "details": []}

    n_verified = 0
    n_passed = 0
    details = []

    for iid, kw in zip(instruction_ids, kwargs_list):
        if iid not in CONSTRAINT_REGISTRY:
            details.append({"id": iid, "verified": False, "passed": None})
            continue

        try:
            cls = CONSTRAINT_REGISTRY[iid]
            instr = cls(iid)
            clean_kw = {k: v for k, v in kw.items() if v is not None}
            instr.build_description(**clean_kw)
            args = instr.get_instruction_args()
            if args and "prompt" in args:
                instr.build_description(prompt=user_prompt)
            passed = bool(instr.check_following(response))
            n_verified += 1
            if passed:
                n_passed += 1
            details.append({"id": iid, "verified": True, "passed": passed})
        except Exception as e:
            # Constraint is in registry but verification errored (e.g. unexpected
            # kwarg name from a newer IFTrain variant). Treat as unverified so
            # it doesn't count against the example.
            details.append({"id": iid, "verified": False, "passed": None,
                             "error": str(e)})

    if n_verified > 0 and n_passed < n_verified:
        status = "failed"
    elif n_verified > 0 and n_passed == n_verified:
        status = "passed"
    else:
        status = "unverified"

    return {"status": status, "n_total": len(instruction_ids),
            "n_verified": n_verified, "n_passed": n_passed, "details": details}


if os.path.exists(VALIDATION_CACHE):
    print(f"Loading cached validation results from:\n  {VALIDATION_CACHE}")
    with open(VALIDATION_CACHE) as f:
        validation_results = json.load(f)
    print(f"Loaded {len(validation_results)} records")
else:
    print("Validating teacher responses...")
    validation_results = []
    for i, result in enumerate(teacher_results):
        vr = validate_response(result)
        vr["key"] = result["key"]
        vr["idx"] = result.get("idx", i)
        validation_results.append(vr)
        if (i + 1) % 500 == 0:
            print(f"  Validated {i + 1}/{len(teacher_results)}")

    with open(VALIDATION_CACHE, "w") as f:
        json.dump(validation_results, f)
    print(f"Validation results saved to:\n  {VALIDATION_CACHE}")

# Summary
status_counts = Counter(v["status"] for v in validation_results)
print(f"\nValidation summary ({len(validation_results)} examples):")
for status, count in sorted(status_counts.items()):
    print(f"  {status:12s}: {count:5d} ({count / len(validation_results) * 100:.1f}%)")

verified_ids = set()
unverified_ids = set()
for vr in validation_results:
    for d in vr["details"]:
        (verified_ids if d["verified"] else unverified_ids).add(d["id"])

print(f"\nConstraint coverage:")
print(f"  In registry:     {len(verified_ids)} types")
print(f"  NOT in registry: {len(unverified_ids)} types")
if unverified_ids:
    print(f"  Unverified (sample): {sorted(unverified_ids)[:10]}")

In [ ]:
# Filtering policy:
#   KEEP if status == 'passed'  (all verifiable constraints satisfied)
#   KEEP if status == 'unverified' (no constraint could be checked, but response looks OK)
#   REJECT if status == 'failed' or 'empty'
#   REJECT if response is too short

keep_indices = []
reject_reasons = Counter()

for i, (result, vr) in enumerate(zip(teacher_results, validation_results)):
    if vr["status"] == "empty":
        reject_reasons["empty_response"] += 1
        continue
    if vr["status"] == "failed":
        reject_reasons["constraint_failed"] += 1
        continue
    if len(result["response"].split()) < MIN_RESPONSE_WORDS:
        reject_reasons["too_short"] += 1
        continue
    keep_indices.append(i)

filtered_results = [teacher_results[i] for i in keep_indices]

print(f"Filtered dataset: {len(filtered_results)} / {len(teacher_results)} kept "
      f"({len(filtered_results) / len(teacher_results) * 100:.1f}%)")
print(f"\nRejection breakdown:")
for reason, count in reject_reasons.most_common():
    print(f"  {reason}: {count}")

## 5. Distilled SFT Dataset Creation

Format the validated (prompt, response) pairs into a HuggingFace `Dataset` using the Llama-3-8B-Instruct chat template for supervised fine-tuning.

In [ ]:
from transformers import AutoTokenizer as _AT

student_tokenizer = _AT.from_pretrained(STUDENT_MODEL, use_fast=True)
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token


def format_for_sft(result):
    """Format a validated teacher result as a Llama-3-Instruct chat turn."""
    messages = [
        {"role": "user",      "content": result["user_prompt"]},
        {"role": "assistant", "content": result["response"]},
    ]
    return student_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


sft_records = []
skipped_long = 0

for r in filtered_results:
    text = format_for_sft(r)
    tok_len = len(student_tokenizer.encode(text))
    if tok_len > MAX_SEQ_LEN:
        skipped_long += 1
        continue
    sft_records.append({
        "text": text,
        "key": r["key"],
        "user_prompt": r["user_prompt"],
        "response": r["response"],
        "constraint": r["constraint"],
        "n_tokens": tok_len,
    })

sft_dataset = Dataset.from_list(sft_records)

print(f"SFT dataset: {len(sft_dataset)} examples")
print(f"  Skipped (too long for {MAX_SEQ_LEN} tokens): {skipped_long}")
token_lengths = sft_dataset["n_tokens"]
print(f"  Token lengths — mean: {np.mean(token_lengths):.0f}, "
      f"median: {int(np.median(token_lengths))}, max: {max(token_lengths)}")
print(f"\nSample SFT text (first 500 chars):\n{sft_records[0]['text'][:500]}")

In [ ]:
SFT_DATASET_PATH = os.path.join(SAVE_DIR, "sft_dataset")
sft_dataset.save_to_disk(SFT_DATASET_PATH)
print(f"SFT dataset saved to: {SFT_DATASET_PATH}")

sft_jsonl = os.path.join(SAVE_DIR, "sft_dataset.jsonl")
with open(sft_jsonl, "w") as f:
    for rec in sft_records:
        f.write(json.dumps({"text": rec["text"]}) + "\n")
print(f"SFT JSONL backup saved to: {sft_jsonl}")

## 6. Llama-3-8B-Instruct QLoRA Fine-Tuning

Load `Meta-Llama-3-8B-Instruct` in 4-bit NF4 quantization and attach LoRA adapters.  
Fine-tune with TRL’s `SFTTrainer` on the distilled dataset, then merge the adapter back into the base weights.

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {STUDENT_MODEL} with 4-bit quantization...")
student_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL, use_fast=True)
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token
student_model.config.pad_token_id = student_tokenizer.pad_token_id

student_model = prepare_model_for_kbit_training(student_model)
print(f"Model loaded. Trainable params before LoRA: "
      f"{sum(p.numel() for p in student_model.parameters() if p.requires_grad):,}")

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Avoid stacking multiple LoRA adapters if the cell is re-run
if not hasattr(student_model, "peft_config"):
    student_model = get_peft_model(student_model, lora_config)

student_model.print_trainable_parameters()

ADAPTER_DIR = os.path.join(SAVE_DIR, "llama3_8b_lora_adapter")
MERGED_DIR  = os.path.join(SAVE_DIR, "llama3_8b_merged")

training_args = TrainingArguments(
    output_dir=os.path.join(SAVE_DIR, "training_output"),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    fp16=False,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    report_to="none",
    seed=SEED,
)

from datasets import load_from_disk
from transformers import Trainer, DataCollatorForLanguageModeling

sft_train = load_from_disk(SFT_DATASET_PATH)

# Tokenize once up-front and use the standard HF Trainer API

def _tokenize_batch(batch):
    return student_tokenizer(
        batch["text"],
        max_length=MAX_SEQ_LEN,
        truncation=True,
        padding="max_length",
    )

print("Tokenizing SFT dataset for Trainer...")

tokenized_train = sft_train.map(
    _tokenize_batch,
    batched=True,
    remove_columns=sft_train.column_names,
)

tokenized_train.set_format(type="torch")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=student_tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=student_model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
)

print(f"Trainer ready — {len(tokenized_train)} training examples, {NUM_TRAIN_EPOCHS} epochs")
effective_batch = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS
print(f"Effective batch size: {effective_batch}")
print(f"Steps per epoch: ~{len(tokenized_train) // effective_batch}")

In [ ]:
print("Starting QLoRA training...")
train_start = time.time()
train_result = trainer.train()
train_time = time.time() - train_start

print(f"\nTraining complete in {train_time / 60:.1f} min")
print(f"  Final loss: {train_result.training_loss:.4f}")
for k, v in train_result.metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# Save LoRA adapter to Drive
trainer.save_model(ADAPTER_DIR)
student_tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to: {ADAPTER_DIR}")

# Free training resources
del student_model, trainer
gc.collect()
torch.cuda.empty_cache()

# Merge LoRA adapter into base model on CPU (saves GPU memory)
print("Merging LoRA weights into base model (on CPU)...")
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL, torch_dtype=torch.bfloat16, device_map="cpu",
)
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(MERGED_DIR)
student_tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to: {MERGED_DIR}")

del base_model, merged_model
gc.collect()

## 7. IFBench Evaluation

Load the merged fine-tuned model with vLLM and evaluate on the official IFBench test set (300 prompts, 58 constraint types).  
Uses the [allenai/IFBench](https://github.com/allenai/IFBench) evaluation harness.  
Reports **strict** and **loose** accuracy at both the prompt and instruction level.

In [ ]:
if __name__ == '__main__':
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    print(f"Loading merged model from: {MERGED_DIR}")
    eval_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR, use_fast=True)

    eval_llm = LLM(
        model=MERGED_DIR,
        dtype="float16",
        gpu_memory_utilization=0.95,
        max_model_len=4096,
        enforce_eager=False,
    )

    eval_sampling = SamplingParams(
        temperature=0,
        top_k=20,
        max_tokens=EVAL_MAX_TOKENS,
    )

    print("Evaluation model loaded!")

In [ ]:
print("Loading IFBench test set...")
ifbench_test = load_dataset("allenai/IFBench_test", split="train")
print(f"Test set: {len(ifbench_test)} examples")

constraint_counts = Counter()
for ex in ifbench_test:
    for iid in ex["instruction_id_list"]:
        constraint_counts[iid] += 1
total_constraints = sum(constraint_counts.values())
print(f"Unique constraint types: {len(constraint_counts)}")
print(f"Total constraints: {total_constraints}")
print(f"Mean constraints per prompt: {total_constraints / len(ifbench_test):.1f}")


def build_eval_prompt(user_prompt):
    return eval_tokenizer.apply_chat_template(
        [{"role": "user", "content": user_prompt}],
        tokenize=False, add_generation_prompt=True,
    )


eval_prompts = [build_eval_prompt(ex["prompt"]) for ex in ifbench_test]

print(f"\nGenerating responses for {len(eval_prompts)} test prompts...")
gen_start = time.time()
eval_outputs = eval_llm.generate(eval_prompts, eval_sampling)
eval_gen_time = time.time() - gen_start

eval_total_tokens = sum(len(o.outputs[0].token_ids) for o in eval_outputs)
eval_throughput = eval_total_tokens / eval_gen_time if eval_gen_time > 0 else 0

print(f"\nGeneration complete:")
print(f"  Time:       {eval_gen_time / 60:.2f} min")
print(f"  Tokens:     {eval_total_tokens:,}")
print(f"  Throughput: {eval_throughput:.1f} tok/s")

eval_responses = [o.outputs[0].text.strip() for o in eval_outputs]

# Save in IFBench evaluation format
EVAL_RESPONSE_FILE = os.path.join(SAVE_DIR, "eval_responses.jsonl")
with open(EVAL_RESPONSE_FILE, "w") as f:
    for ex, resp in zip(ifbench_test, eval_responses):
        f.write(json.dumps({"prompt": ex["prompt"], "response": resp}) + "\n")

EVAL_TEST_FILE = os.path.join(SAVE_DIR, "IFBench_test.jsonl")
with open(EVAL_TEST_FILE, "w") as f:
    for ex in ifbench_test:
        f.write(json.dumps({
            "key": ex["key"], "prompt": ex["prompt"],
            "instruction_id_list": ex["instruction_id_list"],
            "kwargs": ex["kwargs"],
        }) + "\n")

print(f"Responses saved to: {EVAL_RESPONSE_FILE}")
print(f"Test data saved to: {EVAL_TEST_FILE}")

In [ ]:
# Re-pin the IFBench evaluation modules.
# After cell 20 loaded IFEvalG, sys.modules may contain the wrong
# instructions_registry. We explicitly evict and reload from /content/IFBench
# so evaluation_lib uses the correct 58 test-constraint verifiers.
import importlib

for mod_name in ["instructions_registry", "instructions", "evaluation_lib"]:
    sys.modules.pop(mod_name, None)

# Ensure /content/IFBench is first in path
if "/content/IFBench" in sys.path:
    sys.path.remove("/content/IFBench")
sys.path.insert(0, "/content/IFBench")

import instructions_registry
import evaluation_lib

print(f"Evaluation registry: {len(instructions_registry.INSTRUCTION_DICT)} IFBench test constraint types")

inputs = evaluation_lib.read_prompt_list(EVAL_TEST_FILE)
prompt_to_response = evaluation_lib.read_prompt_to_response_dict(EVAL_RESPONSE_FILE)

print(f"Loaded {len(inputs)} test inputs, {len(prompt_to_response)} responses")
missing = [inp.prompt[:80] for inp in inputs if inp.prompt not in prompt_to_response]
if missing:
    print(f"WARNING: {len(missing)} prompts have no matching response!")
else:
    print("All prompts matched.")

# --- Strict evaluation ---
print("\nRunning STRICT evaluation...")
strict_outputs = [
    evaluation_lib.test_instruction_following_strict(inp, prompt_to_response)
    for inp in inputs
]
strict_prompt_correct = sum(o.follow_all_instructions for o in strict_outputs)
strict_prompt_acc = strict_prompt_correct / len(strict_outputs)
strict_instr_total = sum(len(o.follow_instruction_list) for o in strict_outputs)
strict_instr_correct = sum(sum(o.follow_instruction_list) for o in strict_outputs)
strict_instr_acc = strict_instr_correct / strict_instr_total

print(f"  Strict prompt-level:  {strict_prompt_acc:.4f} ({strict_prompt_correct}/{len(strict_outputs)})")
print(f"  Strict instr-level:   {strict_instr_acc:.4f} ({strict_instr_correct}/{strict_instr_total})")

evaluation_lib.write_outputs(os.path.join(SAVE_DIR, "eval_results_strict.jsonl"), strict_outputs)

# --- Loose evaluation ---
print("\nRunning LOOSE evaluation...")
loose_outputs = [
    evaluation_lib.test_instruction_following_loose(inp, prompt_to_response)
    for inp in inputs
]
loose_prompt_correct = sum(o.follow_all_instructions for o in loose_outputs)
loose_prompt_acc = loose_prompt_correct / len(loose_outputs)
loose_instr_total = sum(len(o.follow_instruction_list) for o in loose_outputs)
loose_instr_correct = sum(sum(o.follow_instruction_list) for o in loose_outputs)
loose_instr_acc = loose_instr_correct / loose_instr_total

print(f"  Loose prompt-level:   {loose_prompt_acc:.4f} ({loose_prompt_correct}/{len(loose_outputs)})")
print(f"  Loose instr-level:    {loose_instr_acc:.4f} ({loose_instr_correct}/{loose_instr_total})")

evaluation_lib.write_outputs(os.path.join(SAVE_DIR, "eval_results_loose.jsonl"), loose_outputs)

In [ ]:
# Per-constraint accuracy breakdown (loose evaluation)
constraint_total = defaultdict(int)
constraint_correct = defaultdict(int)

for o in loose_outputs:
    for iid, followed in zip(o.instruction_id_list, o.follow_instruction_list):
        constraint_total[iid] += 1
        if followed:
            constraint_correct[iid] += 1

constraint_rows = []
for cid in sorted(constraint_total.keys()):
    total = constraint_total[cid]
    correct = constraint_correct[cid]
    constraint_rows.append({
        "constraint": cid, "correct": correct,
        "total": total, "accuracy": correct / total,
    })

constraint_df = pd.DataFrame(constraint_rows).sort_values("accuracy", ascending=False)

print("PER-CONSTRAINT ACCURACY (Loose)")
print("=" * 65)
print(constraint_df.to_string(index=False))
print(f"\nConstraints at 100%: {len(constraint_df[constraint_df['accuracy'] == 1.0])}")
print(f"Constraints at   0%: {len(constraint_df[constraint_df['accuracy'] == 0.0])}")

## 8. Summary Metrics & Analysis

In [ ]:
print("\n" + "=" * 65)
print("IFBENCH DISTILLATION EXPERIMENT — FINAL SUMMARY")
print("=" * 65)
print(f"  Teacher model:                  {TEACHER_MODEL}")
print(f"  Student model:                  {STUDENT_MODEL}")
print(f"  Training data:                  allenai/IF_multi_constraints_upto5")
print(f"  Training examples (sampled):    {len(teacher_results)}")
print(f"  Training examples (filtered):   {len(filtered_results)}")
print(f"  Training examples (SFT final):  {len(sft_records)}")
print(f"  LoRA rank / alpha:              {LORA_R} / {LORA_ALPHA}")
print(f"  Training epochs:                {NUM_TRAIN_EPOCHS}")
print(f"  Learning rate:                  {LEARNING_RATE}")
print()
print(f"  IFBench Test Results:")
print(f"    Strict prompt-level accuracy: {strict_prompt_acc:.4f}")
print(f"    Strict instr-level accuracy:  {strict_instr_acc:.4f}")
print(f"    Loose prompt-level accuracy:  {loose_prompt_acc:.4f}  <-- primary metric")
print(f"    Loose instr-level accuracy:   {loose_instr_acc:.4f}")
print()
print(f"  Reference (IFBench leaderboard, loose prompt-level):")
print(f"    OpenAI o3:                69.3%")
print(f"    Qwen2.5 + IF-RLVR:       53.7%")
print(f"    Tulu3 DPO + IF-RLVR:     43.3%")
print(f"    Qwen 3 8B (0-shot):      35.0%")
print(f"    Ours (LoRA distill):      {loose_prompt_acc * 100:.1f}%")
print("=" * 65)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm

fig, ax = plt.subplots(figsize=(12, max(8, len(constraint_df) * 0.3)))

sorted_df = constraint_df.sort_values("accuracy", ascending=True)
colors = [cm.get_cmap("RdYlGn")(v) for v in sorted_df["accuracy"]]

ax.barh(sorted_df["constraint"], sorted_df["accuracy"] * 100,
        color=colors, edgecolor="white", linewidth=0.5)

mean_acc = constraint_df["accuracy"].mean() * 100
ax.axvline(mean_acc, color="navy", linestyle="--", linewidth=1.5,
           label=f"Mean = {mean_acc:.1f}%")
ax.axvline(loose_prompt_acc * 100, color="red", linestyle="-.", linewidth=1.5,
           label=f"Prompt-level = {loose_prompt_acc * 100:.1f}%")

ax.set_xlabel("Accuracy (%)")
ax.set_title(
    f"IFBench Per-Constraint Accuracy\n"
    f"Llama-3-8B-Instruct + QLoRA (Qwen3.5-9B distill)",
    fontweight="bold",
)
ax.set_xlim(0, 105)
ax.legend(fontsize=10)
ax.tick_params(axis="y", labelsize=7)

plt.tight_layout()
fig_path = os.path.join(SAVE_DIR, "per_constraint_accuracy.png")
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to: {fig_path}")

In [ ]:
final_metrics = {
    "experiment": "llama3_8b_lora_qwen35_9b_distill",
    "teacher_model": TEACHER_MODEL,
    "student_model": STUDENT_MODEL,
    "training_data": "allenai/IF_multi_constraints_upto5",
    "train_subset_size": TRAIN_SUBSET_SIZE,
    "train_examples_raw": len(teacher_results),
    "train_examples_filtered": len(filtered_results),
    "train_examples_sft": len(sft_records),
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "num_epochs": NUM_TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "max_seq_len": MAX_SEQ_LEN,
    "strict_prompt_accuracy": round(strict_prompt_acc, 4),
    "strict_instruction_accuracy": round(strict_instr_acc, 4),
    "loose_prompt_accuracy": round(loose_prompt_acc, 4),
    "loose_instruction_accuracy": round(loose_instr_acc, 4),
    "unique_constraint_types_tested": len(constraint_total),
    "eval_total_tokens": eval_total_tokens,
    "eval_throughput_tok_s": round(eval_throughput, 2),
    "eval_gen_time_min": round(eval_gen_time / 60, 3),
}

METRICS_FILE = os.path.join(SAVE_DIR, "final_metrics.json")
with open(METRICS_FILE, "w") as f:
    json.dump(final_metrics, f, indent=2)

CONSTRAINT_FILE = os.path.join(SAVE_DIR, "per_constraint_accuracy.json")
with open(CONSTRAINT_FILE, "w") as f:
    json.dump(constraint_rows, f, indent=2)

print("=== Final Metrics ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

print(f"\nAll artifacts saved to: {SAVE_DIR}")
print(f"  - teacher_generations.json")
print(f"  - validation_results.json")
print(f"  - sft_dataset/ & sft_dataset.jsonl")
print(f"  - llama3_8b_lora_adapter/")
print(f"  - llama3_8b_merged/")
print(f"  - eval_responses.jsonl")
print(f"  - eval_results_strict.jsonl & eval_results_loose.jsonl")
print(f"  - final_metrics.json")
print(f"  - per_constraint_accuracy.json & .png")

In [ ]:
# Show sample evaluation outputs
print("SAMPLE EVALUATION OUTPUTS (first 5)")
print("=" * 65)

for i in range(min(5, len(loose_outputs))):
    o = loose_outputs[i]
    print(f"\n--- Example {i} ---")
    print(f"  Prompt (first 100):  {o.prompt[:100]}...")
    print(f"  Constraints:         {o.instruction_id_list}")
    print(f"  All satisfied:       {o.follow_all_instructions}")
    print(f"  Per-constraint:      {o.follow_instruction_list}")
    print(f"  Response (first 200): {o.response[:200]}...")